In [3]:
# ============================================================
# Cell 1: Tools
# Fault-Tolerance Scaffold Simulator
#
# Classical repetition code:
#   1. Encode one logical bit into N physical bits.
#   2. Each cycle:
#      a. Hardware flips each physical bit with probability p.
#      b. Measurement flips each observed bit with probability q.
#      c. Software decodes by majority vote and corrects.
#   3. After k cycles, count logical failures.
#
# Multi-cycle model: fresh hardware errors are introduced at
# the start of every cycle and applied to the current
# (already-corrected) physical state. This models sustained
# error correction against ongoing hardware noise, which is
# the physically and pedagogically correct picture for
# fault tolerance.
#
# Analytical functions validate the Monte Carlo code for the
# single-cycle case. Multi-cycle analytical results use the
# single-cycle recurrence P_fail(k) = 1 - (1 - p_L)^k.
# ============================================================

import random
import math


# ── Basic helpers ─────────────────────────────────────────────────────────────

def make_odd(n):
    """Force N to be a positive odd integer."""
    n = max(1, int(n))
    return n if n % 2 == 1 else n + 1


def majority_decode(bits):
    """Return the majority bit. Assumes odd number of bits."""
    return 1 if sum(bits) > len(bits) / 2 else 0


def effective_observed_error(p, q):
    """
    Effective bit error rate seen by the decoder when hardware error p
    is followed by measurement error q.

        r = p(1-q) + (1-p)q = p + q - 2pq
    """
    return p + q - 2 * p * q


# ── Single-cycle simulation ───────────────────────────────────────────────────

def one_trial(n, p, q, target=0):
    """
    Run one single-cycle repetition-code trial.

    Returns True if the majority-vote decoder recovers the target bit.
    """
    n = make_odd(n)
    bits = [target] * n

    physical = [1 - b if random.random() < p else b for b in bits]
    measured = [1 - b if random.random() < q else b for b in physical]
    decoded  = majority_decode(measured)

    return decoded == target


def simulate(n=3, p=0.1, q=0.0, trials=10000, target=0, seed=None):
    """
    Monte Carlo estimate of single-cycle logical failure rate.
    """
    n      = make_odd(n)
    p      = min(max(float(p), 0.0), 1.0)
    q      = min(max(float(q), 0.0), 1.0)
    trials = max(1, int(trials))

    if seed is not None:
        random.seed(seed)

    failures = sum(
        0 if one_trial(n=n, p=p, q=q, target=target) else 1
        for _ in range(trials)
    )
    return failures / trials


# ── Multi-cycle simulation ────────────────────────────────────────────────────

def one_multicycle_trial(n, p, q, cycles, target=0):
    """
    Run one multi-cycle repetition-code trial.

    Each cycle:
      1. Hardware flips each physical bit with probability p
         (applied to the current corrected state).
      2. Measurement flips each observed bit with probability q.
      3. Majority-vote decoder corrects the physical state.

    Returns True if the logical bit is recovered correctly after
    all cycles. A single failed correction cycle is a logical failure.
    """
    n      = make_odd(n)
    cycles = max(1, int(cycles))

    # Current physical state — starts as the encoded target.
    state = [target] * n

    for _ in range(cycles):
        # Hardware errors applied to current (corrected) state.
        after_hw = [1 - b if random.random() < p else b for b in state]

        # Measurement errors corrupt what the decoder sees.
        measured = [1 - b if random.random() < q else b for b in after_hw]

        # Majority-vote decoding determines the inferred logical bit.
        decoded = majority_decode(measured)

        # Correct the physical state: reset all bits to the decoded value.
        # This is the software correction step — the decoder's output
        # drives the next cycle's starting state.
        state = [decoded] * n

    # Final logical value is the majority of the last corrected state.
    return majority_decode(state) == target


def simulate_multicycle(n=3, p=0.1, q=0.0, cycles=1,
                        trials=10000, target=0, seed=None):
    """
    Monte Carlo estimate of logical failure rate after k cycles.

    Parameters
    ----------
    n      : repetition code length (forced odd)
    p      : hardware bit-flip probability per cycle
    q      : measurement/readout error probability per cycle
    cycles : number of error-correction cycles
    trials : number of Monte Carlo trials
    target : logical bit to encode (0 or 1)
    seed   : optional RNG seed for reproducibility

    Returns
    -------
    Logical failure rate in [0, 1].
    """
    n      = make_odd(n)
    p      = min(max(float(p), 0.0), 1.0)
    q      = min(max(float(q), 0.0), 1.0)
    cycles = max(1, int(cycles))
    trials = max(1, int(trials))

    if seed is not None:
        random.seed(int(seed))

    failures = sum(
        0 if one_multicycle_trial(n=n, p=p, q=q,
                                  cycles=cycles, target=target) else 1
        for _ in range(trials)
    )
    return failures / trials


# ── Analytical functions (single-cycle exact, multi-cycle recurrence) ─────────

def analytical_logical_failure(n, p, q=0.0):
    """
    Exact single-cycle logical failure probability for an odd-N
    repetition code with hardware error p and measurement error q.

    The decoder sees effective error rate r = p + q - 2pq.
    Logical failure occurs when more than half the measured bits are wrong.
    """
    n = make_odd(n)
    r = effective_observed_error(p, q)
    k_min = n // 2 + 1

    return sum(
        math.comb(n, k) * (r ** k) * ((1 - r) ** (n - k))
        for k in range(k_min, n + 1)
    )


def analytical_multicycle_failure(n, p, q=0.0, cycles=1):
    """
    Approximate logical failure probability after k cycles.

    Uses the single-cycle recurrence:
        P_fail(k) = 1 - (1 - p_L)^k

    where p_L is the single-cycle analytical logical failure rate.

    This approximation treats each cycle as an independent Bernoulli
    trial and is accurate when p_L << 1 (the useful operating regime).
    It slightly overestimates failure probability when p_L is large.

    Parameters
    ----------
    n      : repetition code length (forced odd)
    p      : hardware error probability per cycle
    q      : measurement error probability per cycle
    cycles : number of correction cycles

    Returns
    -------
    Approximate logical failure probability in [0, 1].
    """
    p_L = analytical_logical_failure(n=n, p=p, q=q)
    return 1.0 - (1.0 - p_L) ** cycles


def analytical_curve(n, q=0.0, cycles=1, points=101):
    """
    Return p values and analytical logical failure curve
    for fixed N, q, and number of cycles.
    """
    xs, ys = [], []
    for i in range(points):
        p = i / (points - 1)
        xs.append(p)
        ys.append(analytical_multicycle_failure(n=n, p=p, q=q, cycles=cycles))
    return xs, ys


# ── Resource estimation ───────────────────────────────────────────────────────

def resource_curve(p, q=0.0, cycles=1,
                   n_max=51, targets=None):
    """
    Compute logical failure rate vs repetition code length N
    for fixed hardware error p, measurement error q, and cycle count.

    Returns
    -------
    ns     : list of odd N values from 1 to n_max
    p_L    : list of analytical logical failure rates
    targets: dict {label: threshold} of horizontal reference lines
             e.g. {r'$10^{-2}$': 1e-2, r'$10^{-3}$': 1e-3}
    """
    if targets is None:
        targets = {r'$10^{-2}$': 1e-2,
                   r'$10^{-3}$': 1e-3,
                   r'$10^{-4}$': 1e-4}

    ns  = [n for n in range(1, n_max + 1, 2)]
    pLs = [analytical_multicycle_failure(n=n, p=p, q=q, cycles=cycles)
           for n in ns]

    return ns, pLs


def resource_required_n(p, q=0.0, cycles=1,
                        target_p_L=1e-3, n_max=201):
    """
    Find the minimum odd N such that the analytical logical failure rate
    is at or below target_p_L, for given p, q, and cycle count.

    Returns None if no such N exists within n_max.
    """
    for n in range(1, n_max + 1, 2):
        if analytical_multicycle_failure(n=n, p=p, q=q, cycles=cycles) \
                <= target_p_L:
            return n
    return None


# ── Text utilities ────────────────────────────────────────────────────────────

def text_bar(value, width=40):
    """Simple text progress bar."""
    value  = min(max(value, 0.0), 1.0)
    filled = int(round(value * width))
    return "█" * filled + "░" * (width - filled)


def print_summary(n, p, q, cycles, trials, target, mc_failure, exact_failure):
    """Human-readable single-point summary."""
    r = effective_observed_error(p, q)

    print("Fault-Tolerance Scaffold Simulator")
    print("=" * 44)
    print()
    print("Architecture:")
    print(f"  Logical bit:                 {target}")
    print(f"  Repetition length N:          {n}")
    print(f"  Correction cycles:            {cycles}")
    print(f"  Correctable measured flips:   0 to {n//2}")
    print(f"  Logical-failure flips:        {n//2 + 1} to {n}")
    print()
    print("Error model (per cycle):")
    print(f"  Hardware bit-flip error p:    {p:.4f}")
    print(f"  Measurement/readout error q:  {q:.4f}")
    print(f"  Effective observed error r:   {r:.4f}")
    print()
    print("Results:")
    print(f"  Monte Carlo trials:           {trials}")
    print(f"  MC logical failure:           {mc_failure:.6f}")
    print(f"  Analytical logical failure:   {exact_failure:.6f}")
    print()
    print("Logical failure rate:")
    print(f"  {text_bar(mc_failure)} {mc_failure:.4f}")
    print()
    print("Interpretation:")
    print("  Hardware introduces fresh errors each cycle.")
    print("  Measurement errors corrupt what the software sees.")
    print("  Majority-vote correction runs every cycle.")
    print("  Fault tolerance means correction keeps pace with damage.")


def print_curve_table(n_values, q=0.0, cycles=1):
    """
    Compact table of analytical logical failure at selected p values.
    """
    p_values = [0.01, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

    print()
    print("Analytical logical failure probability")
    print(f"Measurement error q = {q:.3f}   Cycles = {cycles}")
    print("-" * 72)
    header = "N \\ p  " + "".join(f"{p:>10.2f}" for p in p_values)
    print(header)
    print("-" * 72)

    for n in n_values:
        row = f"{n:>5} "
        for p in p_values:
            y = analytical_multicycle_failure(n=n, p=p, q=q, cycles=cycles)
            row += f"{y:>10.4f}"
        print(row)

    print("-" * 72)


In [4]:
# ============================================================
# Cell 2: Compact UI, Three-Panel Plots, and Save Figure
#
# Requires Cell 1 functions:
#   simulate_multicycle()
#   analytical_logical_failure()
#   analytical_multicycle_failure()
#   resource_curve()
#   resource_required_n()
#   print_curve_table()
#
# Panels:
#   (a) Majority-vote curves: p -> p_L for several N at selected q, cycles
#   (b) Operating region heatmap: p, q for selected N, cycles
#   (c) Resource estimation: N -> p_L for selected p, q, cycles
#       with hardware error reference lines for multiple p values
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML


# ── Small helpers ─────────────────────────────────────────────────────────────

def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def clean_odd_n(n):
    n = int(n)
    if n < 1:
        n = 1
    if n % 2 == 0:
        n += 1
    return n


def add_label_background(labels, alpha=0.88):
    for txt in labels:
        txt.set_bbox(dict(facecolor="white", edgecolor="none",
                          alpha=alpha, pad=0.35))


def compact_summary_html(n, p, q, cycles, trials, target,
                          mc_failure, exact_failure):
    return f"""
    <div style="
        font-family: Arial, sans-serif;
        max-width: 980px;
        border: 1px solid #ddd;
        border-radius: 8px;
        padding: 12px 14px;
        margin-bottom: 10px;
        background: #fafafa;
    ">
      <div style="display: flex; gap: 36px; align-items: flex-start;">
        <div style="flex: 1;">
          <b>Architecture</b><br>
          String length: <b>N = {n}</b><br>
          Correction cycles: <b>k = {cycles}</b><br>
          Input logical bit: <b>{target}</b><br>
          QEC / decoder: <b>majority vote</b>
        </div>
        <div style="flex: 1;">
          <b>Error model (per cycle)</b><br>
          Hardware bit-flip probability: <b>p = {p:.4f}</b><br>
          Measurement/readout error: <b>q = {q:.4f}</b><br>
          Monte Carlo trials: <b>{trials}</b>
        </div>
      </div>
      <hr style="border: none; border-top: 1px solid #ddd; margin: 10px 0;">
      <div>
        Selected operating point ({cycles} cycle{'s' if cycles > 1 else ''}):
        Monte Carlo logical failure = <b>{mc_failure:.6g}</b> &nbsp;|&nbsp;
        Analytical logical failure = <b>{exact_failure:.6g}</b>
        &nbsp;<span style="color:#888; font-size:0.88em;">
        (analytical uses single-cycle recurrence)</span>
      </div>
    </div>
    """


# ── Three-panel plotting ──────────────────────────────────────────────────────

def make_three_panel_plots(
    selected_n=7,
    selected_q=0.0,
    selected_p=0.05,
    selected_cycles=1,
    p_max=0.10,
    q_max=0.10,
    target_levels=None,
    grid_points=141,
    log_floor=1e-9,
    n_max_resource=51,
    resource_p_values=None,
    save_prefix=None,
    save_png=True,
    save_pdf=True,
    show_plot=True
):
    """
    Three-panel figure.

    Panel (a): p -> p_L line plot for several N at selected q and cycles.
    Panel (b): p,q heatmap for selected N and cycles.
    Panel (c): N -> p_L resource estimation curve for selected p, q, cycles,
               with lines for multiple hardware error rates.
    """
    if target_levels is None:
        target_levels = [1e-2, 1e-3, 1e-4]

    if resource_p_values is None:
        resource_p_values = [0.01, 0.03, 0.05, 0.10, 0.15]

    target_levels = sorted([
        float(x) for x in target_levels
        if x is not None and float(x) > 0
    ])

    p_max = max(float(p_max), 1e-4)
    q_max = max(float(q_max), 1e-4)

    fig, axes = plt.subplots(
        1, 3,
        figsize=(13.8, 4.6),
        constrained_layout=True
    )

    # ── Panel (a): majority-vote curves ──────────────────────────────────────

    ax = axes[0]
    n_values = [1, 3, 5, 7, 9, 15, 31]
    ps_full  = [i / 250 for i in range(251)]

    for n in n_values:
        ys = [
            analytical_multicycle_failure(
                n=n, p=p, q=selected_q, cycles=selected_cycles)
            for p in ps_full
        ]
        lw = 1.8 if n == selected_n else 1.1
        ax.plot(ps_full, ys, linewidth=lw, label=f"N={n}")

    ax.plot(ps_full, ps_full,
            linestyle="--", color="black", linewidth=1.1,
            label=r"$p_L=p$")
    ax.axvline(0.5, linestyle=":", color="red",
               linewidth=1.4, label=r"$p=0.5$")

    cycle_label = f"k={selected_cycles}"
    ax.set_title(
        f"(a) Majority-vote curves\nq={selected_q:.2f}, {cycle_label}",
        fontsize=9)
    ax.set_xlabel("hardware error p")
    ax.set_ylabel(r"logical failure $p_L$")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_box_aspect(1)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=7, loc="upper left", framealpha=0.9)

    # ── Panel (b): heatmap ───────────────────────────────────────────────────

    ax = axes[1]
    ps = [p_max * i / (grid_points - 1) for i in range(grid_points)]
    qs = [q_max * i / (grid_points - 1) for i in range(grid_points)]

    Z = []
    improvement_boundary = []
    max_value = log_floor

    for q in qs:
        row, improve_row = [], []
        for p in ps:
            val = analytical_multicycle_failure(
                n=selected_n, p=p, q=q, cycles=selected_cycles)
            val_for_log = max(val, log_floor)
            row.append(val_for_log)
            improve_row.append(val - p)
            max_value = max(max_value, val_for_log)
        Z.append(row)
        improvement_boundary.append(improve_row)

    vmin = min(log_floor, min(target_levels) if target_levels else log_floor)
    vmax = min(1.0, max(1e-2, max_value))

    im = ax.imshow(
        Z,
        origin="lower",
        extent=[0, p_max, 0, q_max],
        aspect="auto",
        norm=LogNorm(vmin=vmin, vmax=vmax)
    )
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(r"$p_L$", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    visible_targets = [
        level for level in target_levels
        if level >= vmin and level <= max_value
    ]
    if visible_targets:
        cs = ax.contour(ps, qs, Z,
                        levels=visible_targets,
                        colors="red", linewidths=1.7)
        labels = ax.clabel(cs, inline=True, fontsize=10,
                           fmt={level: f"{level:.0e}"
                                for level in visible_targets})
        add_label_background(labels)

    try:
        cs2 = ax.contour(ps, qs, improvement_boundary,
                         levels=[0.0],
                         colors="black", linewidths=1.5, linestyles="--")
        labels2 = ax.clabel(cs2, inline=True, fontsize=10,
                            fmt={0.0: r"$p_L=p$"},
                            manual=[(0.60 * p_max, 0.60 * q_max)])
        add_label_background(labels2)
    except Exception:
        pass

    if p_max >= 0.5:
        ax.axvline(0.5, color="red", linestyle=":", linewidth=1.5)

    ax.set_title(
        f"(b) Operating region\nN={selected_n}, {cycle_label}",
        fontsize=9)
    ax.set_xlabel("hardware error p")
    ax.set_ylabel("measurement error q")
    ax.set_xlim(0, p_max)
    ax.set_ylim(0, q_max)
    ax.set_box_aspect(1)

    # ── Panel (c): resource estimation ───────────────────────────────────────

    ax = axes[2]

    colors = plt.cm.viridis(
        [i / max(len(resource_p_values) - 1, 1)
         for i in range(len(resource_p_values))]
    )

    for i, rp in enumerate(resource_p_values):
        ns, pLs = resource_curve(
            p=rp, q=selected_q,
            cycles=selected_cycles,
            n_max=n_max_resource
        )
        lw   = 2.0 if abs(rp - selected_p) < 1e-6 else 1.2
        ls   = "-"  if abs(rp - selected_p) < 1e-6 else "--"
        ax.semilogy(ns, pLs,
                    color=colors[i], linewidth=lw,
                    linestyle=ls, label=f"p={rp:.2f}")

    # Horizontal reference lines for target logical error rates.
    ref_styles = [
        (target_levels[0] if len(target_levels) > 0 else 1e-2,
         "red",    ":",  r"$10^{-2}$"),
        (target_levels[1] if len(target_levels) > 1 else 1e-3,
         "orange", ":",  r"$10^{-3}$"),
        (target_levels[2] if len(target_levels) > 2 else 1e-4,
         "green",  ":",  r"$10^{-4}$"),
    ]
    for level, color, ls, label in ref_styles:
        ax.axhline(level, color=color, linestyle=ls,
                   linewidth=1.2, alpha=0.8, label=label)

    ax.set_title(
        f"(c) Resource estimation\nq={selected_q:.2f}, {cycle_label}",
        fontsize=9)
    ax.set_xlabel("repetition code length N")
    ax.set_ylabel(r"logical failure $p_L$  (log scale)")
    ax.set_xlim(1, n_max_resource)
    ax.set_ylim(log_floor, 1.0)
    ax.set_box_aspect(1)
    ax.grid(True, which="both", alpha=0.2)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.9)

    if save_prefix:
        if save_png:
            fig.savefig(f"{save_prefix}.png", dpi=300, bbox_inches="tight")
        if save_pdf:
            fig.savefig(f"{save_prefix}.pdf", bbox_inches="tight")

    if show_plot:
        plt.show()
    else:
        plt.close(fig)

    return fig


# ── Widget definitions ────────────────────────────────────────────────────────

main_style   = {"description_width": "95px"}
target_style = {"description_width": "165px"}
save_style   = {"description_width": "110px"}
res_style    = {"description_width": "135px"}

main_layout   = widgets.Layout(width="260px")
target_layout = widgets.Layout(width="390px")
save_layout   = widgets.Layout(width="320px")
res_layout    = widgets.Layout(width="320px")

# Operating point controls.
n_input = widgets.BoundedIntText(
    value=7, min=1, max=999, step=2,
    description="N bits",
    continuous_update=False,
    style=main_style, layout=main_layout
)

p_input = widgets.BoundedFloatText(
    value=0.05, min=0.0, max=1.0, step=0.01,
    description="p",
    continuous_update=False,
    style=main_style, layout=main_layout
)

q_input = widgets.BoundedFloatText(
    value=0.01, min=0.0, max=0.5, step=0.01,
    description="q",
    continuous_update=False,
    style=main_style, layout=main_layout
)

cycles_input = widgets.BoundedIntText(
    value=1, min=1, max=1000, step=1,
    description="cycles k",
    continuous_update=False,
    style=main_style, layout=main_layout
)

trials_input = widgets.BoundedIntText(
    value=10000, min=1, max=10000000, step=1000,
    description="trials",
    continuous_update=False,
    style=main_style, layout=main_layout
)

target_dropdown = widgets.Dropdown(
    options=[0, 1], value=0,
    description="target",
    style=main_style, layout=main_layout
)

seed_box = widgets.IntText(
    value=7,
    description="seed",
    style=main_style, layout=main_layout
)

use_seed_box = widgets.Checkbox(
    value=True, description="use seed",
    layout=main_layout
)

# Heatmap axis controls.
pmax_input = widgets.BoundedFloatText(
    value=0.10, min=0.001, max=1.0, step=0.01,
    description="p max",
    continuous_update=False,
    style=main_style, layout=main_layout
)

qmax_input = widgets.BoundedFloatText(
    value=0.10, min=0.001, max=0.5, step=0.01,
    description="q max",
    continuous_update=False,
    style=main_style, layout=main_layout
)

# Logical error target contours.
target1_input = widgets.FloatLogSlider(
    value=1e-2, base=10, min=-9, max=-1, step=1,
    description="logical target 1",
    readout_format=".0e",
    continuous_update=False,
    style=target_style, layout=target_layout
)

target2_input = widgets.FloatLogSlider(
    value=1e-3, base=10, min=-9, max=-1, step=1,
    description="logical target 2",
    readout_format=".0e",
    continuous_update=False,
    style=target_style, layout=target_layout
)

target3_input = widgets.FloatLogSlider(
    value=1e-4, base=10, min=-9, max=-1, step=1,
    description="logical target 3",
    readout_format=".0e",
    continuous_update=False,
    style=target_style, layout=target_layout
)

# Resource estimation controls.
nmax_resource_input = widgets.BoundedIntText(
    value=51, min=3, max=301, step=2,
    description="N max (resource)",
    continuous_update=False,
    style=res_style, layout=res_layout
)

resource_p_text = widgets.Text(
    value="0.01, 0.03, 0.05, 0.10, 0.15",
    description="p values (resource)",
    style=res_style, layout=res_layout
)

# Save controls.
save_prefix_input = widgets.Text(
    value="fault_tolerance_scaffold_figure_v2.0",
    description="save prefix",
    style=save_style, layout=save_layout
)

save_png_box = widgets.Checkbox(
    value=True, description="save PNG",
    layout=save_layout
)

save_pdf_box = widgets.Checkbox(
    value=True, description="save PDF",
    layout=save_layout
)

run_button = widgets.Button(
    description="Run simulator",
    button_style="primary",
    layout=widgets.Layout(width="160px")
)

output = widgets.Output()


# ── Main callback ─────────────────────────────────────────────────────────────

def run_demo(_=None):
    with output:
        clear_output(wait=True)

        n      = clean_odd_n(n_input.value)
        p      = clamp(float(p_input.value), 0.0, 1.0)
        q      = clamp(float(q_input.value), 0.0, 0.5)
        cycles = max(1, int(cycles_input.value))
        trials = max(1, int(trials_input.value))
        target = target_dropdown.value
        seed   = seed_box.value if use_seed_box.value else None

        p_max = clamp(float(pmax_input.value), 0.001, 1.0)
        q_max = clamp(float(qmax_input.value), 0.001, 0.5)

        target_levels = sorted([
            float(target1_input.value),
            float(target2_input.value),
            float(target3_input.value)
        ])

        n_max_res = max(3, int(nmax_resource_input.value))
        if n_max_res % 2 == 0:
            n_max_res += 1

        try:
            resource_p_values = [
                float(x.strip())
                for x in resource_p_text.value.split(",")
                if x.strip()
            ]
        except ValueError:
            resource_p_values = [0.01, 0.03, 0.05, 0.10, 0.15]

        save_prefix = save_prefix_input.value.strip() or None

        # Keep UI honest for odd N.
        n_input.value = n

        mc_failure   = simulate_multicycle(
            n=n, p=p, q=q, cycles=cycles,
            trials=trials, target=target, seed=seed
        )
        exact_failure = analytical_multicycle_failure(
            n=n, p=p, q=q, cycles=cycles
        )

        display(HTML(compact_summary_html(
            n=n, p=p, q=q, cycles=cycles, trials=trials,
            target=target,
            mc_failure=mc_failure, exact_failure=exact_failure
        )))

        make_three_panel_plots(
            selected_n=n,
            selected_q=q,
            selected_p=p,
            selected_cycles=cycles,
            p_max=p_max,
            q_max=q_max,
            target_levels=target_levels,
            grid_points=141,
            log_floor=1e-9,
            n_max_resource=n_max_res,
            resource_p_values=resource_p_values,
            save_prefix=save_prefix,
            save_png=save_png_box.value,
            save_pdf=save_pdf_box.value,
            show_plot=True
        )

        if save_prefix:
            saved = []
            if save_png_box.value:
                saved.append(f"{save_prefix}.png")
            if save_pdf_box.value:
                saved.append(f"{save_prefix}.pdf")
            if saved:
                display(HTML(
                    "<div style='font-family: Arial; margin: 8px 0;'>"
                    f"<b>Saved figure:</b> {', '.join(saved)}"
                    "</div>"
                ))

        display(HTML("""
        <div style="
            font-family: Arial, sans-serif;
            max-width: 980px;
            margin-top: 8px;
            padding: 10px 12px;
            border-left: 4px solid #555;
            background: #f7f7f7;
            line-height: 1.45;
        ">
        <b>Interpretation.</b>
        <b>(a)</b> Each curve shows how logical failure rate changes with
        hardware error for a given code length N and number of correction
        cycles k. Below the threshold (left of the dashed p<sub>L</sub>=p line),
        adding more physical bits suppresses logical errors. Above it, more bits
        make things worse. <b>(b)</b> The heatmap shows the joint operating region
        in hardware error p and measurement error q for the selected N and k.
        Red contours mark target logical failure rates.
        <b>(c)</b> The resource curve answers: how many physical bits N do I need
        to reach a target logical error rate, given my hardware error p?
        Each line is a different p value; the selected p is shown solid.
        Horizontal dotted lines mark the logical error targets.
        Running more cycles shifts all curves upward &mdash; fault tolerance
        requires the per-cycle error rate to stay below threshold.
        </div>
        """))

        display(HTML(
            "<div style='font-family: Arial; margin-top: 6px;'>"
            "<b>Logical-error targets used:</b> "
            + ", ".join([f"{x:.0e}" for x in target_levels])
            + f" &nbsp;|&nbsp; <b>Cycles:</b> {cycles}"
            + "</div>"
        ))

        print_curve_table(
            n_values=[1, 3, 5, 7, 9, 15, 31],
            q=q,
            cycles=cycles
        )


run_button.on_click(run_demo)


# ── Layout ────────────────────────────────────────────────────────────────────

main_controls = widgets.VBox([
    widgets.HTML("<b>Selected operating point</b>"),
    n_input,
    p_input,
    q_input,
    cycles_input,
    trials_input
])

run_controls = widgets.VBox([
    widgets.HTML("<b>Run settings</b>"),
    target_dropdown,
    use_seed_box,
    seed_box,
    run_button
])

heatmap_controls = widgets.VBox([
    widgets.HTML("<b>Heatmap view</b>"),
    pmax_input,
    qmax_input,
    target1_input,
    target2_input,
    target3_input
])

resource_controls = widgets.VBox([
    widgets.HTML("<b>Resource estimation</b>"),
    nmax_resource_input,
    resource_p_text
])

save_controls = widgets.VBox([
    widgets.HTML("<b>Save figure</b>"),
    save_prefix_input,
    save_png_box,
    save_pdf_box
])

ui_top    = widgets.HBox([main_controls, run_controls, heatmap_controls])
ui_bottom = widgets.HBox([resource_controls, save_controls])

display(HTML("""
<div style="font-family: Arial, sans-serif; max-width: 980px;">
<h3>Fault-Tolerance Scaffold Simulator</h3>
<p>
Classical repetition code with hardware error <b>p</b> and measurement
error <b>q</b> applied <b>every cycle</b>, corrected by majority vote.
Panel (a) shows the broad threshold behavior across code lengths.
Panel (b) zooms into the p,q operating region for the selected N and
cycle count. Panel (c) shows how many physical bits are needed to reach
a target logical error rate — the resource estimation curve.
</p>
</div>
"""))

display(ui_top)
display(ui_bottom)
display(output)

run_demo()


Output()